---
last_verified: 2026-08-04
tool_version: n/a
---

# Terraform State, Backends, and Modules — interactive exploration

This notebook walks through three core Terraform concepts: state management, remote backends, and reusable modules. Each section includes runnable examples and verification steps.



## Purpose

Terraform state tracks the real-world resources your configuration manages. Backends control where that state is stored and how operations are executed. Modules package reusable infrastructure components. Understanding all three is essential for any Terraform workflow beyond a single-file local experiment.

## 1 — Terraform State

Terraform state maps your configuration to real resources. By default it is stored in a local file called `terraform.tfstate`. Every `terraform plan` and `terraform apply` reads and updates this file. When the state file is missing or out of date, Terraform cannot determine what already exists and may attempt to recreate resources.

In [ ]:
%%bash

# Initialize a working directory for state exploration
WORKDIR=$(mktemp -d)
cat > "$WORKDIR/main.tf" <<'EOF'
terraform {
  required_version = ">= 1.0"
}

resource "null_resource" "demo" {
  triggers = {
    stamp = timestamp()
  }
}
EOF
echo "Created demo config in $WORKDIR"
ls -la "$WORKDIR"

### What just happened

A minimal Terraform configuration was written to a temporary directory. It declares a single `null_resource` — a resource that does nothing except exist in state. This is useful for demonstrating state behavior without provisioning real infrastructure.

In [ ]:
%%bash

# Show that state does not exist yet (no resources tracked)
cd "$WORKDIR"
terraform plan 2>&1 | head -20

### State locking

When multiple operators run Terraform against the same state, conflicts can arise. Remote backends provide state locking to prevent concurrent writes. Local backends do not offer locking by default — a second `terraform apply` on the same state file can corrupt it.

## 2 — Terraform Backends

A backend determines where Terraform stores its state file and how it executes operations. The `local` backend is the default. Remote backends — such as S3, Azure Blob Storage, Google Cloud Storage, Terraform Cloud, and Consul — store state externally and can provide locking and collaboration features.

In [ ]:
%%bash

# Show the default backend behavior
cd "$WORKDIR"
terraform init 2>&1 | tail -5
echo "---"
cat "$WORKDIR/.terraform/terraform.tfstate" 2>/dev/null | python3 -m json.tool 2>/dev/null | head -10 || echo "State file not yet created — run plan or apply first"

# Clean up
rm -rf "$WORKDIR"

### Backend configuration example

A backend is configured in the `terraform` block. The example below shows an S3 backend configuration. The actual values (bucket name, region, key) would need to match a real S3 bucket.

In [ ]:
# Example S3 backend configuration (not executed — no real AWS resources)
# terraform {
#   backend "s3" {
#     bucket = "my-terraform-state-bucket"
#     key    = "project/terraform.tfstate"
#     region = "us-east-1"
#   }
# }

### Backend selection guidance

- **Local** — suitable for single-operator, experimental, or CI ephemeral environments.
- **S3 + DynamoDB** — common for AWS-based teams; DynamoDB provides state locking.
- **Terraform Cloud/Enterprise** — managed state with API access, run triggers, and policy enforcement.
- **Azure Blob / GCS** — equivalent managed options for their respective clouds.

The choice depends on team size, cloud provider, and collaboration requirements.

## 3 — Terraform Modules

A module is a reusable collection of Terraform configuration. Modules accept inputs, expose outputs, and encapsulate resources. They can be sourced from the local filesystem, the Terraform Registry, or a private Git repository.

In [ ]:
%%bash

# Create a local module structure
WORKDIR=$(mktemp -d)
MODULE_DIR="$WORKDIR/modules/simple-ec2"
mkdir -p "$MODULE_DIR"

cat > "$MODULE_DIR/variables.tf" <<'EOF'
variable "name_prefix" {
  type    = string
  default = "demo"
}

variable "instance_type" {
  type    = string
  default = "t3.micro"
}
EOF

cat > "$MODULE_DIR/outputs.tf" <<'EOF'
output "resource_name" {
  value = var.name_prefix
}
EOF

cat > "$MODULE_DIR/main.tf" <<'EOF'
resource "null_resource" "this" {
  triggers = {
    name = var.name_prefix
    type = var.instance_type
  }
}
EOF

echo "Module created at $MODULE_DIR"
ls -la "$MODULE_DIR"

### Using the module

A root configuration calls the module with a `module` block, passing inputs and reading outputs.

In [ ]:
# Example of calling the module from a root config (not executed)
# module "demo_instance" {
#   source        = "./modules/simple-ec2"
#   name_prefix   = "production-web"
#   instance_type = "t3.small"
# }
#
# output "instance_name" {
#   value = module.demo_instance.resource_name
# }

| Source type | Example | Use case |
|---|---|---|
| Local path | `./modules/vpc` | Organization-internal reusable components |
| Registry | `terraform-aws-modules/vpc/aws` | Community-tested, versioned modules |
| Registry public | `hashicorp/consul/aws` | Official HashiCorp modules |

Version pinning is recommended for registry sources to prevent unexpected breaking changes.


## Verify

To confirm understanding of these concepts:

1. **State**: Run `terraform plan` in a directory with a config and observe that Terraform creates a state file after the first apply.
2. **Backend**: Change the backend configuration from `local` to `s3` (with real credentials) and observe that `terraform init` migrates state.
3. **Module**: Create a local module with variables and outputs, then call it from a root config with different input values.

Each step reinforces how state, backends, and modules interact in a real Terraform workflow.

In [ ]:
%%bash

# Quick summary of the three concepts
echo "Terraform State    — tracks resources in a state file"
echo "Terraform Backends — control where state is stored and how operations run"
echo "Terraform Modules  — reusable, parameterized configuration packages"
echo ""
echo "Together they enable: shared state, team collaboration, and reusable infrastructure patterns."